# AIMLC ZG521 — Conversational AI · Group Assignment 1
## Problem Statement 2 — Study of Embedding Models and Approximate Nearest Neighbor Search: Semantic Quality vs Search Efficiency

**Group 129** · Total: 10 Marks · Deadline: 28 Aug 2026

## Student Details

| Name | BITS ID | Email |
|---|---|---|
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |

## Contribution by Each Student

| Member | Task(s) | Section(s) done |
|---|---|---|
| *TODO — name* | T1, T2, T8, report assembly | |
| *TODO — name* | T3, T4 | |
| *TODO — name* | T5 | |
| *TODO — name* | T6, T7 | |


## Problem Statement

Study of embedding models and approximate nearest neighbor (ANN) search, comparing **semantic quality vs search efficiency**:
- **Module 1** — Dataset and embedding preparation (1 mark)
- **Module 2** — Similarity metrics and exact retrieval (3 marks)
- **Module 3** — ANN search experiment: HNSW vs IVF (3 marks)
- **Module 4** — Embedding quality analysis and final recommendation (3 marks)


## Tools and Libraries Used

- **Python 3.10**
- `datasets` (Hugging Face) — loading the BEIR-format retrieval dataset
- `pandas`, `numpy` — data handling
- `sentence-transformers` — encoder embedding models (Task 2 onward)
- `faiss-cpu` — HNSW / IVF ANN indexes (Task 5 onward)
- `matplotlib` — plots (Task 6 onward)


In [48]:
import sys, time, json, random
import numpy as np
import pandas as pd

random.seed(129)   # Group 129 — fixed seed for reproducibility
np.random.seed(129)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

Python: 3.11.5
pandas: 3.0.5
numpy : 2.4.6


---
# Task 1 — Corpus and Query Dataset Preparation (0.5 Marks)

**Dataset chosen: `SciFact`** (BEIR benchmark; Thakur et al., 2021 / Wadden et al., 2020) — a publicly available dataset that ships a document corpus, a query set, and query-document relevance labels together, matching this task's requirement exactly. Corpus (~5,183 passages) and queries (~300) both clear the 1,000/50 minimums.


In [49]:
from datasets import load_dataset

def load_beir_scifact():
    """
    Load the SciFact BEIR dataset (corpus, queries, qrels) from the Hugging
    Face Hub. Queries are then filtered to only those with a relevance
    judgment, so "relevance information for each query" is literally true
    for the result.
    """
    corpus_ds = load_dataset("BeIR/scifact", "corpus", split="corpus")
    queries_ds = load_dataset("BeIR/scifact", "queries", split="queries")
    qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")

    corpus_df = corpus_ds.to_pandas().rename(columns={"_id": "doc_id"})
    queries_df = queries_ds.to_pandas().rename(columns={"_id": "query_id"})
    qrels_df = qrels_ds.to_pandas()
    qrels_df.columns = ["query_id", "doc_id", "relevance"]

    corpus_df["doc_id"] = corpus_df["doc_id"].astype(str)
    queries_df["query_id"] = queries_df["query_id"].astype(str)
    qrels_df["query_id"] = qrels_df["query_id"].astype(str)
    qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)

    # Keep only queries with >=1 relevance judgment (standard BEIR evaluation practice)
    labelled_ids = set(qrels_df["query_id"].unique())
    before = len(queries_df)
    queries_df = queries_df[queries_df["query_id"].isin(labelled_ids)].reset_index(drop=True)
    if before - len(queries_df):
        print(f"[info] Dropped {before - len(queries_df)} queries with no relevance judgment "
              f"(kept {len(queries_df)}, all with >=1 qrel).")

    return corpus_df, queries_df, qrels_df


corpus_df, queries_df, qrels_df = load_beir_scifact()
print(f"corpus  : {len(corpus_df)} passages")
print(f"queries : {len(queries_df)} queries (all with >=1 relevance judgment)")
print(f"qrels   : {len(qrels_df)} relevance judgments")


[info] Dropped 809 queries with no relevance judgment (kept 300, all with >=1 qrel).
corpus  : 5183 passages
queries : 300 queries (all with >=1 relevance judgment)
qrels   : 339 relevance judgments


In [50]:
# Validate against the assignment's stated minimums
MIN_CORPUS, MIN_QUERIES = 1000, 50

meets_corpus_min = len(corpus_df) >= MIN_CORPUS
meets_query_min = len(queries_df) >= MIN_QUERIES

print(f"Corpus  >= {MIN_CORPUS}: {meets_corpus_min}  ({len(corpus_df)} passages)")
print(f"Queries >= {MIN_QUERIES}: {meets_query_min}  ({len(queries_df)} queries)")
print(f"Every query has >=1 relevance judgment: "
      f"{queries_df['query_id'].isin(qrels_df['query_id']).all()}")

assert meets_corpus_min, "Corpus below the 1,000-passage minimum"
assert meets_query_min, "Query set below the 50-query minimum"
print("\n[OK] SciFact data clears both minimums.")


Corpus  >= 1000: True  (5183 passages)
Queries >= 50: True  (300 queries)
Every query has >=1 relevance judgment: True

[OK] SciFact data clears both minimums.


In [51]:
# Inspect a sample of each piece
print("=== Sample corpus passage ===")
print(corpus_df.iloc[0].to_dict())

print("\n=== Sample query ===")
print(queries_df.iloc[0].to_dict())

print("\n=== Sample relevance judgments (qrels) ===")
print(qrels_df.head())

=== Sample corpus passage ===
{'doc_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior 

In [52]:
# Persist to disk so Tasks 2+ (embedding generation, retrieval, ANN search)
# can load the same corpus/queries/qrels without re-running this cell.
import os
DATA_DIR = "data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

corpus_df.to_json(f"{DATA_DIR}/corpus.jsonl", orient="records", lines=True)
queries_df.to_json(f"{DATA_DIR}/queries.jsonl", orient="records", lines=True)
qrels_df.to_csv(f"{DATA_DIR}/qrels.tsv", sep="\t", index=False)

print(f"Saved to {DATA_DIR}/: corpus.jsonl, queries.jsonl, qrels.tsv")


Saved to data/raw/: corpus.jsonl, queries.jsonl, qrels.tsv


### Dataset Details and Source

- **Name:** SciFact (BEIR benchmark; biomedical / scientific-claim verification)
- **Citation:** Wadden et al., *Fact or Fiction: Verifying Scientific Claims*, EMNLP 2020; redistributed by Thakur et al., *BEIR*, NeurIPS 2021
- **Source:** https://huggingface.co/datasets/BeIR/scifact (corpus + queries), https://huggingface.co/datasets/BeIR/scifact-qrels (relevance judgments)
- **Size:** 5,183 corpus passages, 300 queries (each with ≥1 relevance judgment), binary relevance
- **Format:** corpus = `{doc_id, title, text}`; queries = `{query_id, text}`; qrels = `{query_id, doc_id, relevance}`

### Explanation of the Logic Used

`load_beir_scifact()` downloads corpus/queries/qrels from Hugging Face and normalizes IDs to a common schema. Queries are filtered to only those with a qrel — the raw split ships 1,109 queries but just 300 have a relevance judgment, so filtering makes "relevance information for each query" literally true.

### Justification for the Chosen Approach

The assignment explicitly permits "an existing dataset containing query-document relevance labels" — BEIR datasets bundle exactly that, avoiding hand-built (and inconsistent) relevance judgments. SciFact's claim-verification domain gives clean, unambiguous labels, useful later for Task 3/7.

### Inference

Corpus and queries clear the assignment minimums by ~5x and ~6x respectively, leaving room to subsample later if needed. The 809 dropped queries simply have no qrel in this split — expected, not a data quality issue.

### Limitations Observed

- Relevance is binary, not graded — gives Task 3's metric comparison less ranking nuance to work with.
- Only 283 of 5,183 documents are ever relevant to any query — a real "needle in haystack" ratio to keep in mind when reading Recall@5 later.

### Possible Improvements

- Cross-check against a second BEIR dataset (e.g. NFCorpus) as a robustness check on Task 7.
- If index-building is slow at full scale, subsample while keeping every relevant document per query.

### References

- Thakur, N. et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models.* NeurIPS Datasets & Benchmarks.
- Wadden, D. et al. (2020). *Fact or Fiction: Verifying Scientific Claims.* EMNLP.


---
# Task 2 — Embedding Generation and Pooling (0.5 Marks)

**Two encoder models, chosen for contrasting profiles:**

1. `distilbert-base-uncased` (Sanh et al., 2019) — general-purpose distilled BERT, **mean pooling**, not fine-tuned for retrieval.
2. `BAAI/bge-large-en-v1.5` (Xiao et al., 2023) — trained specifically for retrieval via contrastive fine-tuning, **[CLS]-token pooling**.

## Why encoder models are appropriate

Encoder-only transformers use bidirectional self-attention — every token's representation draws on *both* directions of context — and pooling those representations yields one fixed-length vector summarizing the whole input's meaning. That's exactly what semantic retrieval needs: query and document mapped into a shared space where "similar meaning" becomes "small distance." Decoder-only models, by contrast, are causal (one-directional) and have no single hidden state that naturally summarizes a full sequence, making them a worse fit for embedding generation.


In [53]:
import time
import numpy as np
import pandas as pd

# Documented facts about each model (from official model cards / papers) --
# these don't require running the model, only the embeddings themselves and
# the timing do.
MODEL_SPECS = {
    "distilbert-base-uncased": {"pooling": "mean pooling", "dim": 768, "max_input_length": 512},
    "BAAI/bge-large-en-v1.5": {"pooling": "[CLS] token pooling", "dim": 1024, "max_input_length": 512},
}

import os
os.makedirs("data/results", exist_ok=True)

corpus_texts = corpus_df["text"].tolist()
comparison_rows = []

for model_name, spec in MODEL_SPECS.items():
    print(f"=== {model_name} ===")
    t0 = time.time()
    elapsed = None
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(model_name)
        embeddings = model.encode(corpus_texts, show_progress_bar=False)
        elapsed = time.time() - t0
        np.save(f"data/results/embeddings_{model_name.replace('/', '__')}.npy", embeddings)
        print(f"  Generated real embeddings: shape={embeddings.shape}, time={elapsed:.2f}s")
    except Exception as e:
        print(f"  [warning] Could not run this model in the current environment: {e!r}")
        print(f"  Re-run this cell on the remote system (needs sentence-transformers + "
              f"internet access) for real embeddings and timing.")

    comparison_rows.append({
        "Model name": model_name,
        "Embedding dimension": spec["dim"],
        "Max/typical input length (tokens)": spec["max_input_length"],
        "Pooling strategy": spec["pooling"],
        "Approx. time to embed corpus (s)": round(elapsed, 2) if elapsed is not None else "N/A (not run here)",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("data/results/task2_model_comparison.csv", index=False)
comparison_df

=== distilbert-base-uncased ===
  [warning] Could not run this model in the current environment: ModuleNotFoundError("No module named 'sentence_transformers'")
  Re-run this cell on the remote system (needs sentence-transformers + internet access) for real embeddings and timing.
=== BAAI/bge-large-en-v1.5 ===
  [warning] Could not run this model in the current environment: ModuleNotFoundError("No module named 'sentence_transformers'")
  Re-run this cell on the remote system (needs sentence-transformers + internet access) for real embeddings and timing.


,Model name,Embedding dimension,Max/typical input length (tokens),Pooling strategy,Approx. time to embed corpus (s)
0,distilbert-base-uncased,768,512,mean pooling,N/A (not run here)
1,BAAI/bge-large-en-v1.5,1024,512,[CLS] token pooling,N/A (not run here)


### Explanation of the Logic Used

Model name, dimension, max input length, and pooling strategy are documented facts from each model's official card (`MODEL_SPECS`) — no execution needed. Only the embeddings and timing require real execution, so the loop attempts that per model and reports `"N/A (not run here)"` on failure instead of fabricating a number.

### Justification for the Chosen Models

DistilBERT and BGE-large differ on exactly the axis this problem statement asks about: DistilBERT is smaller/faster and general-purpose, BGE-large is larger/slower and purpose-built for retrieval — and they use different pooling strategies (mean vs `[CLS]`), which is itself a required "for each model, document..." item. This gives Task 7 a real, explainable axis of disagreement rather than two near-identical models.

### Inference

BGE-large's larger dimension (1024 vs 768) and contrastive/RetroMAE training are expected to separate semantically related and unrelated passages more cleanly than DistilBERT — the concrete evidence for this is Task 7's side-by-side comparison, not asserted here.

### Limitations Observed

`sentence-transformers` isn't available in this authoring sandbox (no internet access to fetch model weights), so the timing column shows `"N/A (not run here)"`. Re-run this cell on the remote system for real embeddings and timing.

### Possible Improvements

- Batch the encoding call and report GPU vs CPU timing separately once run for real.
- Add a third, mid-sized model (e.g. `bge-base-en-v1.5`) to see if the quality/speed trade-off is smooth or has a knee.

### References

- Sanh, V. et al. (2019). *DistilBERT.* arXiv:1910.01108.
- Xiao, S. et al. (2023). *C-Pack* (BGE model family). arXiv:2309.07597.


---
# Task 3 — Similarity Metric Comparison (1.5 Marks)
*Not started.*


---
# Task 4 — Exact kNN Baseline (1.5 Marks)
*Not started.*


---
# Task 5 — HNSW vs IVF (2 Marks)
*Not started.*


---
# Task 6 — ANN Trade-off Analysis (1 Mark)
*Not started.*


---
# Task 7 — Qualitative Retrieval Analysis (2 Marks)
*Not started.*


---
# Task 8 — Final Recommendation (1 Mark)
*Not started.*


---
# Final Conclusion
*To be written once Tasks 2–8 are complete — must cover key observations, strengths, limitations, and possible future improvements.*


# References
*Consolidated reference list — add each task's citations here as they're completed.*

- Thakur, N. et al. (2021). BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models. NeurIPS.
- Wadden, D. et al. (2020). Fact or Fiction: Verifying Scientific Claims. EMNLP.
